# Signal Extractor — Design Math

We have two wave inputs:

$$
\cos(2\pi f_1 t + \varphi_1), \qquad \cos(2\pi f_2 t + \varphi_2)
$$

We put nominal frequencies $\bar{f}_1$ and $\bar{f}_2$ in the script. Assume actual values deviate from nominal by $\pm \delta f_1$ and $\pm \delta f_2$.

Shorthand:
- $f_s = f_1 + f_2$, $f_d = \lvert f_1 - f_2 \rvert$ — actual sum/diff
- $\bar{f}_s = \bar{f}_1 + \bar{f}_2$, $\bar{f}_d = \lvert \bar{f}_1 - \bar{f}_2 \rvert$ — nominal sum/diff
- $\delta\bar{f}_s = \delta f_1 + \delta f_2$ — worst-case absolute uncertainty (same for sum and diff)


## 1. Take the product

$$
\cos(2\pi f_1 t + \varphi_1)\cos(2\pi f_2 t + \varphi_2)
$$

$$
= \tfrac{1}{2}\Bigl[\cos\!\bigl(2\pi(f_1 + f_2)t + (\varphi_1 + \varphi_2)\bigr) + \cos\!\bigl(2\pi(f_1 - f_2)t + (\varphi_1 - \varphi_2)\bigr)\Bigr]
$$

$$
= \tfrac{1}{2}\cos(2\pi f_s t + \varphi_s) + \tfrac{1}{2}\cos(2\pi f_d t + \varphi_d)
$$

where $\varphi_s = \varphi_1 + \varphi_2$ and $\varphi_d = \varphi_1 - \varphi_2$. The phase is signed — no absolute value.


## 2. Lock by nominal carriers

Lock the product against nominal $\bar{f}_s$ for the sum arm and $\bar{f}_d$ for the diff arm. Each tone in the product splits into two baseband images.

### Sum arm (carrier $\bar{f}_s$)

| component | baseband frequency |
|:--|:-:|
| sum tone, downmix | $\lvert \bar{f}_s - f_s \rvert \;\leq\; \delta\bar{f}_s$ |
| sum tone, upmix | $\bar{f}_s + f_s \;\approx\; 2\bar{f}_s$ |
| diff tone, downmix | $\bar{f}_s - f_d \;\approx\; 2\bar{f}_1$ or $2\bar{f}_2$ |
| diff tone, upmix | $\bar{f}_s + f_d \;\approx\; 2\bar{f}_2$ or $2\bar{f}_1$ |

Keep the first one (near DC). Reject the lowest of the rest:

$$
f_\text{reject}^\text{sum} = \min(2\bar{f}_1,\ 2\bar{f}_2)
$$

### Diff arm (carrier $\bar{f}_d$)

| component | baseband frequency |
|:--|:-:|
| diff tone, downmix | $\lvert \bar{f}_d - f_d \rvert \;\leq\; \delta\bar{f}_s$ |
| diff tone, upmix | $\bar{f}_d + f_d \;\geq\; \max\!\bigl(\bar{f}_d,\ 2\bar{f}_d - \delta\bar{f}_s\bigr)$ |
| sum tone, downmix | $\lvert \bar{f}_d - f_s \rvert \;\approx\; 2\bar{f}_1$ or $2\bar{f}_2$ |
| sum tone, upmix | $\bar{f}_d + f_s \;\approx\; 2\bar{f}_2$ or $2\bar{f}_1$ |

Keep first. Reject lowest of the rest (now the upmix of the diff tone can be small too, so include it):

$$
f_\text{reject}^\text{diff} = \min\!\bigl(2\bar{f}_1,\ 2\bar{f}_2,\ \max(\bar{f}_d,\ 2\bar{f}_d - \delta\bar{f}_s)\bigr)
$$

The $\max(\bar{f}_d, \ldots)$ enforces $f_d \geq 0$ — the actual diff frequency can't go negative even when uncertainty would push it there.


## 3. Filter cutoff for 100× suppression

For a 2-stage one-pole LPF (cascaded), the amplitude response is:

$$
\lvert H(f) \rvert = \frac{1}{1 + (f/f_c)^2}
$$

For 100× amplitude suppression at $f_\text{reject}$:

$$
\frac{1}{1 + (f_\text{reject}/f_c)^2} = \frac{1}{100} \;\;\Longrightarrow\;\; (f_\text{reject}/f_c)^2 = 99
$$

$$
\boxed{\,f_c = \frac{f_\text{reject}}{\sqrt{99}}\,}
$$


## 4. After filter + IQ recombination

After the LPF, both I and Q baseband legs carry the signal at offset $\Delta f = f_d - \bar{f}_d$ (signed; magnitude $\leq \delta\bar{f}_s$):

$$
I_\text{filt}(t) = \tfrac{1}{2}\lvert H(\Delta f)\rvert \cos\!\bigl(2\pi \Delta f\, t + \varphi_d + \arg H(\Delta f)\bigr)
$$
$$
Q_\text{filt}(t) = -\tfrac{1}{2}\lvert H(\Delta f)\rvert \sin\!\bigl(2\pi \Delta f\, t + \varphi_d + \arg H(\Delta f)\bigr)
$$

The IQ block remixes with the nominal carrier $\bar{f}_d$ — using $\cos(A)\cos(B) - \sin(A)\sin(B) = \cos(A+B)$ — to move the baseband signal back up:

$$
y_\text{out}(t) = I_\text{filt}\cos(2\pi \bar{f}_d t) + Q_\text{filt}\sin(2\pi \bar{f}_d t)
$$

$$
= \tfrac{1}{2}\lvert H(\Delta f)\rvert \cos\!\bigl(2\pi(\bar{f}_d + \Delta f) t + \varphi_d + \arg H(\Delta f)\bigr)
$$

Since $\bar{f}_d + \Delta f = f_d$, the output sits at the **actual** difference frequency $f_d$ (not nominal) with phase $\varphi_d$ plus the LPF phase shift. The sign of $\Delta f$ is carried by the I/Q phase relationship — no information lost.


## 5. Phase shift

The LPF imposes phase $\arg H(\Delta f)$ on the signal. For $N = 2$ cascaded stages:

$$
\phi_\text{shift} = -N \arctan(\Delta f / f_c) = -2 \arctan\!\left(\frac{\lvert \bar{f}_d - f_d \rvert}{f_c}\right)
$$

If we can keep the frequency uncertainty small compared to the cutoff — $\lvert \bar{f}_d - f_d \rvert \ll f_c$ — the phase shift is effectively zero.

**Phase coherence is set by the ratio $\lvert \bar{f}_d - f_d \rvert / f_c$.** Bigger $f_c$ relative to uncertainty → smaller phase distortion.


## 6. Limitation: $f_1 \approx f_2$

When $f_1 \approx f_2$, the nominal diff $\bar{f}_d$ is small. In the diff arm, the diff-tone upmix sits at $\approx 2\bar{f}_d$ — on the same magnitude as the uncertainty $\delta\bar{f}_s$.

The cutoff is squeezed from both sides:
- $f_c > \delta\bar{f}_s$ — so the signal passes (Section 5)
- $f_c < 2\bar{f}_d - \delta\bar{f}_s$ — so the upmix is rejected (Section 2)

These constraints collide when $2\bar{f}_d$ approaches $\delta\bar{f}_s$. At $f_1 = f_2$ exactly, $\bar{f}_d = 0$ and the diff-arm is degenerate — the bandpass approach can't separate signal from its own image around DC.
